# ESZA019 — Visão Computacional
## Laboratório 7 — Introdução às redes neurais convolucionais

**Grupo 2**  
**Autores:** Cesar de Jesus, Mariana Chiara e Vinicius de Marchi  
**Docente:** Prof. Celso Setsuo Kurashima  
**Data de realização dos experimentos:** 29 de julho de 2026  
**Data de publicação:** 02 de agosto de 2026

---

### Resumo

Neste laboratório foi construída e treinada uma rede neural convolucional (CNN) para classificar as dez categorias do conjunto CIFAR-10. Primeiro, o filtro Sobel X foi aplicado manualmente a nove imagens para comparar o processamento clássico com filtros aprendidos. Em seguida, uma CNN com duas camadas convolucionais foi treinada por dez épocas no Google Colab com GPU T4.

O modelo original alcançou **76,17% de acurácia no treinamento** e **71,15% no conjunto de teste**, valor confirmado pela matriz de confusão. Os maiores números de falsos positivos ocorreram nas classes preditas **cervo** e **gato**, principalmente por confusões entre animais visualmente semelhantes nas imagens de baixa resolução.

Nos testes com webcam, a câmera foi apontada para imagens padronizadas do mesmo automóvel exibidas em uma tela. Alterações de iluminação, fundo e escala mudaram as previsões, mostrando o efeito de *domain shift*. Uma segunda CNN com aumento de dados atingiu **64,75% como melhor acurácia de validação** dentro das dez épocas. Por fim, um limite de confiança de 60% classificou quatro de dez exemplos como “Predição incerta”, evitando decisões com baixa confiança.


## Sumário

1. Introdução  
2. Objetivos  
3. Fundamentação teórica  
4. Materiais e métodos  
5. Implementação e resultados  
6. Testes com webcam  
7. Desafio prático  
8. Análise e discussão  
9. Conclusões  
10. Estrutura do repositório  
11. Referências  
12. Declaração de uso de IA


## 1. Introdução

No processamento clássico de imagens, o programador escolhe previamente operações como suavização, detecção de bordas e limiarização. Um exemplo é o filtro Sobel, cujos coeficientes são definidos manualmente para destacar variações de intensidade em uma direção específica.

As redes neurais convolucionais utilizam uma operação semelhante, mas os valores dos filtros são aprendidos a partir dos dados. Durante o treinamento, os pesos são ajustados para diminuir o erro de classificação. As primeiras camadas costumam responder a bordas e contrastes; camadas posteriores combinam essas respostas para representar texturas, partes e formas mais complexas.

O CIFAR-10 foi utilizado por conter 60 mil imagens coloridas de 32 × 32 pixels, divididas em dez classes. Embora seja um conjunto pequeno e didático, ele permite observar as etapas principais de um sistema de percepção: preparação dos dados, construção da CNN, treinamento, avaliação, interpretação dos mapas de características e uso do modelo em imagens capturadas por webcam.


## 2. Objetivos

O objetivo geral foi compreender a transição de filtros espaciais definidos manualmente para filtros aprendidos por uma CNN e avaliar o modelo em imagens fora do conjunto de treinamento.

Objetivos específicos:

- aplicar o filtro Sobel X a nove imagens do CIFAR-10, sendo três por integrante;
- normalizar as imagens para o intervalo de 0 a 1;
- construir uma CNN compacta com TensorFlow/Keras;
- treinar o modelo por dez épocas;
- analisar acurácia, perda e matriz de confusão;
- visualizar mapas de características da primeira camada convolucional;
- testar o modelo com imagens adquiridas por webcam;
- observar os efeitos de iluminação, fundo complexo e mudança de escala;
- medir a ordem de grandeza do tempo de inferência;
- implementar aumento de dados;
- aplicar um limite de confiança de 60%;
- documentar resultados, limitações e uso de IA de forma transparente.


## 3. Fundamentação teórica

### 3.1 Convolução e filtro Sobel

Para uma imagem \(I\) e um kernel \(K\), a saída de uma convolução discreta pode ser representada por

\[
S(i,j)=\sum_m\sum_n I(i-m,j-n)K(m,n).
\]

O Sobel X usa o kernel

\[
K_x=
\begin{bmatrix}
-1 & 0 & 1\\
-2 & 0 & 2\\
-1 & 0 & 1
\end{bmatrix},
\]

que responde principalmente a mudanças horizontais de intensidade e, por isso, destaca bordas verticais.

### 3.2 Camadas de uma CNN

Uma camada `Conv2D` aplica vários kernels sobre pequenas regiões da imagem. Os pesos desses kernels são ajustados pela retropropagação. A função ReLU mantém ativações positivas e introduz não linearidade. O `MaxPooling2D` reduz altura e largura dos mapas, diminuindo o custo computacional e tornando a representação menos sensível a pequenos deslocamentos.

Depois da extração de características, a camada `Flatten` transforma os mapas em um vetor. As camadas `Dense` combinam essas características para decidir a classe. A saída `softmax` produz dez probabilidades cuja soma é 1.

### 3.3 Treinamento, perda e regularização

Foi usada a perda `sparse_categorical_crossentropy`, apropriada para rótulos inteiros e classificação multiclasse. O otimizador Adam atualiza os pesos a partir dos gradientes. O `Dropout(0.3)` desliga aleatoriamente 30% das ativações durante o treinamento, ajudando a reduzir a memorização dos dados.

### 3.4 Mudança de domínio e segurança

O *domain shift* ocorre quando os dados usados fora do laboratório diferem dos dados de treinamento. No caso da webcam, resolução, iluminação, fundo, escala, perspectiva e a própria exibição em um monitor alteram a imagem. O aumento de dados tenta simular parte dessas variações. O limite de confiança não corrige a classificação, mas permite rejeitar previsões pouco seguras.


## 4. Materiais e métodos

### 4.1 Materiais

- Google Colab;
- GPU NVIDIA T4;
- TensorFlow 2.20.0 e Keras;
- Python, NumPy, OpenCV, Matplotlib, Seaborn e scikit-learn;
- conjunto de dados CIFAR-10;
- webcam do computador;
- monitor usado para exibir os estímulos;
- quatro imagens padronizadas do mesmo automóvel;
- Jupyter Notebook para documentação.

### 4.2 Configuração experimental

O CIFAR-10 foi carregado pelo TensorFlow. As imagens de treino e teste foram convertidas para `float32` e divididas por 255. O modelo original foi treinado por dez épocas com lotes de 64 imagens.

Para os ensaios com webcam, foram produzidas quatro imagens do mesmo automóvel: condição normal, iluminação alterada, fundo complexo e objeto distante. As imagens foram exibidas em tela cheia, e a webcam foi apontada para o monitor. Portanto, o teste avalia a transferência **CIFAR-10 → câmera/tela**, e não o reconhecimento direto de um automóvel físico.

### 4.3 Sequência do experimento

1. Importar as bibliotecas e confirmar a GPU.
2. Carregar o CIFAR-10.
3. Aplicar Sobel X a nove imagens.
4. Normalizar os dados.
5. Construir a CNN.
6. Treinar por dez épocas.
7. gerar curvas e matriz de confusão.
8. Extrair os mapas da camada `conv_1`.
9. Capturar e classificar quatro condições pela webcam.
10. Treinar uma nova CNN com aumento de dados.
11. Aplicar rejeição para confiança menor que 60%.


In [ ]:
# 1. Bibliotecas e configuração
import time
import io
from base64 import b64decode

import cv2
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf
from PIL import Image
from sklearn.metrics import confusion_matrix
from tensorflow.keras import layers, models

print(f"Versão do TensorFlow: {tf.__version__}")
gpu_ativa = len(tf.config.list_physical_devices("GPU")) > 0
print(f"Aceleração por GPU ativa: {gpu_ativa}")


Na execução do grupo, foi utilizado **TensorFlow 2.20.0** e a GPU T4 foi reconhecida corretamente.


## 5. Implementação e resultados

### 5.1 Filtro Sobel X


In [ ]:
# 2. Convolução clássica e nove exemplos
(x_train_raw, y_train_raw), _ = tf.keras.datasets.cifar10.load_data()

sobel_x = np.array(
    [[-1, 0, 1],
     [-2, 0, 2],
     [-1, 0, 1]],
    dtype=np.float32,
)

integrantes = [
    "Cesar de Jesus",
    "Mariana Chiara",
    "Vinicius de Marchi",
]

class_names_sobel = [
    "Avião", "Automóvel", "Pássaro", "Gato", "Cervo",
    "Cachorro", "Sapo", "Cavalo", "Navio", "Caminhão",
]

indices_selecionados = []
for classe in range(9):
    indice = np.where(y_train_raw.flatten() == classe)[0][0]
    indices_selecionados.append(indice)

plt.figure(figsize=(18, 10))
for i, indice in enumerate(indices_selecionados):
    imagem = x_train_raw[indice]
    imagem_cinza = cv2.cvtColor(imagem, cv2.COLOR_RGB2GRAY)
    bordas = cv2.filter2D(imagem_cinza, -1, sobel_x)

    aluno = integrantes[i // 3]
    classe = class_names_sobel[y_train_raw[indice][0]]

    plt.subplot(3, 6, 2 * i + 1)
    plt.imshow(imagem)
    plt.title(f"{aluno}\nOriginal: {classe}")
    plt.axis("off")

    plt.subplot(3, 6, 2 * i + 2)
    plt.imshow(bordas, cmap="gray")
    plt.title("Filtro Sobel X")
    plt.axis("off")

plt.suptitle("Comparação entre as imagens originais e o filtro Sobel X")
plt.tight_layout()
plt.show()


![Nove exemplos do filtro Sobel X](assets/sobel_9_exemplos.png)

**Questão 1 — Limitação dos filtros manuais.** O Sobel procura apenas variações de intensidade na direção definida pelo kernel. Ele não se adapta sozinho a mudanças de iluminação, sombra, ruído, chuva, orientação ou escala. Além disso, detectar uma borda não significa reconhecer o objeto. Em um robô externo, seriam necessários vários ajustes e etapas adicionais.

**Questão 2 — O que a camada convolucional aprende.** A rede aprende os valores numéricos dos kernels. Eles começam com valores aproximadamente aleatórios e são alterados pela retropropagação para reduzir o erro. Assim, a própria CNN descobre quais padrões ajudam a separar as classes, em vez de receber apenas um filtro definido manualmente.


### 5.2 Carregamento e normalização


In [ ]:
# 3. CIFAR-10 e pré-processamento
(x_train, y_train), (x_test, y_test) = (
    tf.keras.datasets.cifar10.load_data()
)

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

class_names = [
    "Aviao", "Automovel", "Passaro", "Gato", "Cervo",
    "Cachorro", "Sapo", "Cavalo", "Navio", "Caminhao",
]

plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i][0]])
    plt.axis("off")
plt.tight_layout()
plt.show()


![Amostras do CIFAR-10](assets/amostras_cifar10.png)

**Questão 3 — Importância da normalização.** Os pixels passam de 0–255 para 0–1. Isso deixa os cálculos em uma escala menor, reduz oscilações nos gradientes e ajuda o otimizador a ajustar os pesos de modo mais estável.

**Questão 4 — Imagem 4K ligada diretamente a uma camada densa.** Uma imagem CIFAR-10 tem \(32\times32\times3=3.072\) valores. Ligá-los a 128 neurônios exigiria 393.344 parâmetros, contando os vieses. Uma imagem 4K tem \(3.840\times2.160\times3=24.883.200\) valores; a mesma ligação exigiria 3.185.049.728 parâmetros, aproximadamente 12,74 GB apenas para pesos de 32 bits. Isso seria pesado para memória, energia e tempo de inferência de um robô. Por isso, imagens maiores são redimensionadas e processadas por convoluções e *pooling* antes das camadas densas.


### 5.3 Arquitetura da CNN


In [ ]:
# 4. Modelo original
def build_robot_cnn(input_shape=(32, 32, 3), num_classes=10):
    model = models.Sequential([
        layers.Conv2D(
            32, (3, 3), activation="relu", padding="same",
            input_shape=input_shape, name="conv_1",
        ),
        layers.MaxPooling2D((2, 2), name="pool_1"),
        layers.Conv2D(
            64, (3, 3), activation="relu",
            padding="same", name="conv_2",
        ),
        layers.MaxPooling2D((2, 2), name="pool_2"),
        layers.Flatten(name="flatten"),
        layers.Dense(128, activation="relu", name="fc_1"),
        layers.Dropout(0.3, name="dropout"),
        layers.Dense(num_classes, activation="softmax", name="output"),
    ])
    return model


model = build_robot_cnn()
model.summary()


| Camada | Forma de saída | Parâmetros |
|---|---:|---:|
| `conv_1` | 32 × 32 × 32 | 896 |
| `pool_1` | 16 × 16 × 32 | 0 |
| `conv_2` | 16 × 16 × 64 | 18.496 |
| `pool_2` | 8 × 8 × 64 | 0 |
| `flatten` | 4.096 | 0 |
| `fc_1` | 128 | 524.416 |
| `dropout` | 128 | 0 |
| `output` | 10 | 1.290 |
| **Total** |  | **545.098** |

**Questão 5 — Arquitetura da rede.** As convoluções ficam no início para procurar características locais, como bordas, cores e formas. O `MaxPooling` reduz o tamanho dos mapas e deixa o processamento mais leve. Depois, o `Flatten` organiza as características em uma sequência e a camada densa combina essas informações para escolher a classe. Se a camada densa viesse primeiro, a organização espacial seria perdida muito cedo e a quantidade de parâmetros aumentaria bastante.


### 5.4 Treinamento do modelo original


In [ ]:
# 5. Compilação e treinamento
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history = model.fit(
    x_train,
    y_train,
    epochs=10,
    validation_data=(x_test, y_test),
    batch_size=64,
)


![Curvas do treinamento original](assets/curvas_treinamento.png)

| Época | Acurácia treino | Perda treino | Acurácia validação | Perda validação |
|---:|---:|---:|---:|---:|
| 1 | 42,70% | 1,5733 | 55,52% | 1,2564 |
| 2 | 56,48% | 1,2237 | 61,62% | 1,0817 |
| 3 | 62,21% | 1,0703 | 65,88% | 0,9644 |
| 4 | 65,80% | 0,9691 | 68,09% | 0,9129 |
| 5 | 68,16% | 0,8996 | 68,72% | 0,8938 |
| 6 | 69,83% | 0,8510 | 69,71% | 0,8715 |
| 7 | 71,86% | 0,7936 | 69,34% | 0,8832 |
| 8 | 73,86% | 0,7406 | 70,20% | 0,8592 |
| 9 | 74,86% | 0,7062 | 71,11% | 0,8435 |
| 10 | **76,17%** | **0,6637** | **71,15%** | **0,8426** |

**Questão 6 — Perda e sobreajuste.** A função de perda mede o erro entre as probabilidades previstas e a classe correta. Durante o experimento, a perda de treino diminuiu e a acurácia aumentou. A validação também melhorou, mas passou a variar menos nas últimas épocas. Se a perda de treino continuar caindo e a de validação começar a subir, ocorre *overfitting*: a rede memoriza o treino e pode falhar em imagens novas, como as capturadas pelo robô.


### 5.5 Matriz de confusão

![Matriz de confusão](assets/matriz_confusao.png)

A diagonal principal contém 7.115 acertos em 10.000 imagens, resultando em **71,15% de acurácia**. Navio (832 acertos), automóvel (817), caminhão (809) e cavalo (800) foram reconhecidos com maior facilidade. Gato (483), pássaro (504) e cachorro (580) apresentaram mais dificuldade.

**Questão 7 — Maiores falsos positivos.** Somando os valores fora da diagonal por coluna, as classes preditas com mais falsos positivos foram **cervo (513)** e **gato (391)**. Muitos pássaros foram classificados como cervos (154), e gatos e cachorros foram bastante confundidos entre si. As imagens têm apenas 32 × 32 pixels, e animais podem compartilhar cores, texturas, poses e fundos naturais.

**Questão 8 — Segurança do robô.** Os erros não possuem necessariamente a mesma gravidade. Confundir automóvel com caminhão ainda identifica um veículo/obstáculo grande, então uma frenagem pode continuar sendo adequada. Confundir cachorro com caminhão perde uma informação importante sobre o tipo e o comportamento do obstáculo. A matriz permite localizar essas confusões por classe e decidir quais erros precisam ser reduzidos antes do uso real.


### 5.6 Mapas de características


In [ ]:
# 6. Saídas da primeira camada convolucional
dummy_input = tf.zeros((1, 32, 32, 3))
_ = model(dummy_input)

activation_model = tf.keras.Model(
    inputs=model.layers[0].input,
    outputs=model.get_layer("conv_1").output,
)

sample_input = np.expand_dims(x_test[0], axis=0)
feature_maps = activation_model.predict(sample_input)

plt.figure(figsize=(12, 6))
for i in range(16):
    plt.subplot(4, 4, i + 1)
    plt.imshow(feature_maps[0, :, :, i], cmap="viridis")
    plt.axis("off")
    plt.title(f"Canal {i + 1}")
plt.suptitle("Mapas aprendidos pela camada conv_1")
plt.show()


![Mapas de características](assets/mapas_caracteristicas.png)

**Questão 9 — Características simples e complexas.** Os canais da primeira camada destacam padrões simples, como bordas, cores e contrastes. Nas camadas mais profundas, esses padrões são combinados. Por isso, a última camada convolucional tende a responder a características mais complexas, como partes e formatos aproximados dos objetos.


## 6. Testes com webcam

### 6.1 Código de captura, classificação e tempo


In [ ]:
# 7. Captura de um frame no Google Colab
from IPython.display import Javascript, display
from google.colab.output import eval_js


def capture_webcam_frame():
    javascript = Javascript('''
    async function takePhoto() {
      const area = document.createElement('div');
      const video = document.createElement('video');
      const botao = document.createElement('button');

      botao.textContent = 'Capturar Frame e Classificar';
      botao.style.padding = '10px 20px';
      botao.style.marginTop = '10px';

      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      document.body.appendChild(area);
      area.appendChild(video);
      area.appendChild(botao);
      video.srcObject = stream;
      await video.play();
      await new Promise((resolve) => botao.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getTracks().forEach(track => track.stop());
      area.remove();
      return canvas.toDataURL('image/jpeg', 0.8);
    }
    ''')
    display(javascript)
    dados = eval_js("takePhoto()")
    binario = b64decode(dados.split(",")[1])
    return np.array(Image.open(io.BytesIO(binario)))


raw_frame = capture_webcam_frame()
img_resized = cv2.resize(raw_frame, (32, 32))
img_tensor = np.expand_dims(
    img_resized.astype("float32") / 255.0,
    axis=0,
)

inicio = time.perf_counter()
prediction = model.predict(img_tensor, verbose=0)
tempo_ms = (time.perf_counter() - inicio) * 1000

class_idx = np.argmax(prediction[0])
confidence = prediction[0][class_idx]

plt.figure(figsize=(6, 6))
plt.imshow(raw_frame)
plt.title(
    f"Predição: {class_names[class_idx]} "
    f"({100 * confidence:.1f}%)\n"
    f"Inferência: {tempo_ms:.1f} ms"
)
plt.axis("off")
plt.show()


### 6.2 Condições ensaiadas

As quatro imagens abaixo são os estímulos exibidos no monitor:

| Condição | Estímulo |
|---|---|
| Normal | [abrir imagem](assets/estimulo_normal.png) |
| Iluminação alterada | [abrir imagem](assets/estimulo_iluminacao.png) |
| Fundo complexo | [abrir imagem](assets/estimulo_fundo_complexo.png) |
| Mudança de escala | [abrir imagem](assets/estimulo_escala.png) |

### 6.3 Resultados

![Painel dos testes com webcam](assets/painel_webcam.png)

| Condição | Predição | Confiança | Avaliação |
|---|---|---:|---|
| normal | caminhão | 84,4% | errou a classe específica, mas reconheceu um veículo |
| iluminação alterada | automóvel | 65,8% | acertou |
| fundo complexo | caminhão | 77,2% | errou a classe específica |
| objeto distante | pássaro | 75,6% | errou e mudou de categoria |

[Abrir vídeo-resumo das quatro capturas](assets/video_resumo_webcam.mp4)

O vídeo acima foi montado a partir das quatro capturas estáticas para facilitar a visualização. Ele não representa uma filmagem contínua da webcam.


In [ ]:
# Visualização opcional do vídeo-resumo no Jupyter
from IPython.display import Video, display

display(Video("assets/video_resumo_webcam.mp4", width=700))


**Questão 10 — *Domain shift*.** O mesmo automóvel recebeu três classes diferentes. Os fatores principais foram iluminação e exposição da câmera, fundo complexo, tamanho do objeto, perspectiva e perda de detalhes ao reduzir a captura para 32 × 32 pixels. Além disso, a câmera observou uma imagem exibida no monitor, acrescentando brilho, contraste e possíveis padrões da tela que não existem no CIFAR-10.

**Questão 11 — Latência e distância percorrida.** Para \(v=2\,\text{m/s}\) e \(t=500\,\text{ms}=0,5\,\text{s}\),

\[
d=v\,t=2\times0,5=1\,\text{m}.
\]

O robô percorreria **1 metro “às cegas”**. Uma CNN maior pode melhorar a acurácia, mas normalmente exige mais processamento. O modelo deve ser escolhido considerando a distância de frenagem e o tempo total de captura, pré-processamento, inferência e comando dos motores.

### Questões adicionais da Parte 8

**Tempo de inferência a 30 FPS.** Nas execuções registradas da webcam, o Keras indicou aproximadamente **4–6 ms por imagem** para a etapa interna de inferência, abaixo do orçamento de 33 ms por ciclo. Porém, esse número não inclui todo o tempo de abertura da câmera, transferência JavaScript/Python, redimensionamento e envio do comando. O código com `time.perf_counter()` permite medir o tempo de `model.predict` no computador utilizado.

**Ilusão da acurácia controlada.** O conjunto de teste do CIFAR-10 é parecido com o conjunto de treino. A webcam produz imagens com outra resolução, iluminação, fundo e escala. Por isso, 71,15% no CIFAR-10 não significa a mesma acurácia no mundo real. Nos quatro ensaios, apenas a condição de iluminação alterada foi classificada exatamente como automóvel.


## 7. Desafio prático

### 7.1 Aumento de dados


In [ ]:
# 8. Nova CNN com aumento de dados
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

model_aug = models.Sequential([
    layers.Input(shape=(32, 32, 3)),
    data_augmentation,
    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(10, activation="softmax"),
])

model_aug.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history_aug = model_aug.fit(
    x_train,
    y_train,
    epochs=10,
    validation_data=(x_test, y_test),
)


![Curvas do modelo com aumento de dados](assets/curvas_aumento_dados.png)

![Comparação entre os modelos](assets/comparacao_modelos.png)

| Modelo | Acurácia final de treino | Acurácia final de validação | Melhor validação |
|---|---:|---:|---:|
| CNN original | 76,17% | 71,15% | 71,15% |
| CNN com aumento de dados | 61,74% | 64,67% | 64,75% (época 7) |

O aumento de dados tornou o treinamento mais difícil. Em apenas dez épocas, o modelo aumentado ficou abaixo do original. Isso não significa que a técnica seja inadequada: rotações, ampliações e inversões aumentam a variedade das entradas, e o modelo pode precisar de mais épocas ou ajustes. Também seria necessário repetir os quatro testes de webcam com `model_aug` para afirmar se houve melhora fora do CIFAR-10.

### 7.2 Limite de confiança


In [ ]:
# 9. Rejeição de previsões com confiança menor que 60%
predictions = model_aug.predict(x_test[:10], verbose=0)

for i, prediction in enumerate(predictions):
    indice = np.argmax(prediction)
    confianca = prediction[indice]

    if confianca < 0.60:
        resultado = "Predição incerta"
    else:
        resultado = class_names[indice]

    print(
        f"Imagem {i + 1}: {resultado} "
        f"({confianca * 100:.1f}%)"
    )


![Resultado do limite de confiança](assets/limite_confianca.png)

Das dez imagens, quatro ficaram abaixo de 60% e foram marcadas como **Predição incerta**. As demais receberam uma classe. A rejeição evita que o sistema apresente como certa uma escolha de baixa confiança, mas o valor de 60% ainda precisa ser calibrado para a aplicação real.


## 8. Análise e discussão

O experimento mostrou a diferença entre extrair uma característica específica com Sobel e aprender vários filtros com uma CNN. O modelo original aprendeu o conjunto CIFAR-10 e alcançou 71,15% de acurácia, mas a matriz de confusão revelou que o valor geral não é suficiente para avaliar segurança. Confusões entre animais foram frequentes, enquanto veículos e navios tiveram maior número de acertos.

Os testes com webcam demonstraram uma limitação ainda mais importante. Mesmo utilizando o mesmo automóvel, iluminação, fundo e escala alteraram a predição. A condição normal foi classificada como caminhão e o carro distante como pássaro. Isso confirma que uma CNN pode apresentar boa métrica no conjunto de teste e ainda falhar quando a origem da imagem muda.

Principais limitações:

- o CIFAR-10 possui resolução muito baixa;
- o conjunto de teste foi usado também como validação, como no roteiro didático;
- os estímulos foram exibidos em um monitor, não eram objetos físicos;
- não foi medida a latência completa até um atuador;
- a confiança da `softmax` não é garantia de acerto;
- o limite de 60% foi escolhido como regra de segurança, mas não foi calibrado;
- o modelo com aumento de dados foi treinado por apenas dez épocas;
- os quatro testes com webcam não permitem estimar uma acurácia estatística real.

Possíveis melhorias:

- coletar imagens reais da própria câmera;
- separar treino, validação e teste;
- usar mais épocas com parada antecipada;
- comparar diferentes intensidades de aumento de dados;
- calibrar as probabilidades e o limite de rejeição;
- usar uma arquitetura leve treinada previamente, como MobileNet;
- medir captura, pré-processamento, inferência e comando de forma completa;
- testar em vídeo e com obstáculos físicos.


## 9. Conclusões

O laboratório atingiu o objetivo de construir e avaliar uma CNN compacta para o CIFAR-10. O filtro Sobel mostrou como um kernel manual destaca bordas, enquanto as camadas convolucionais aprenderam diferentes mapas de características durante o treinamento.

O modelo original atingiu 71,15% de acurácia no conjunto de teste. A matriz de confusão mostrou maior dificuldade entre animais e permitiu localizar falsos positivos que ficariam escondidos em uma única métrica. Nos testes com webcam, o mesmo automóvel recebeu diferentes classificações, comprovando a influência do *domain shift*.

O aumento de dados não superou o modelo original nas dez épocas realizadas, mas apresentou uma forma de expor a rede a variações. O limite de confiança marcou quatro de dez previsões como incertas e mostrou uma estratégia simples de rejeição. Conclui-se que acurácia, latência e confiança devem ser analisadas juntas antes de utilizar uma CNN em um robô real.


## 10. Estrutura do repositório

```text
Lab7_Grupo2/
├── Relatorio_Lab7_Grupo2.ipynb
├── README.md
├── requirements.txt
├── gerar_figuras_lab7.py
└── assets/
    ├── amostras_cifar10.png
    ├── comparacao_modelos.png
    ├── curvas_aumento_dados.png
    ├── curvas_treinamento.png
    ├── estimulo_escala.png
    ├── estimulo_fundo_complexo.png
    ├── estimulo_iluminacao.png
    ├── estimulo_normal.png
    ├── limite_confianca.png
    ├── mapas_caracteristicas.png
    ├── matriz_confusao.png
    ├── painel_webcam.png
    ├── sobel_9_exemplos.png
    ├── video_resumo_webcam.mp4
    ├── webcam_escala.png
    ├── webcam_fundo_complexo.png
    ├── webcam_iluminacao.png
    └── webcam_normal.png
```


## 11. Referências

1. KURASHIMA, C. S. **ESZA019 — Visão Computacional: Laboratório 7 — Introdução às redes CNN**. Universidade Federal do ABC, 2026.
2. SZELISKI, R. **Computer Vision: Algorithms and Applications**. 2. ed. Springer, 2022. Cap. 5.4. DOI: 10.1007/978-3-030-34372-9.
3. KRIZHEVSKY, A. **Learning Multiple Layers of Features from Tiny Images**. University of Toronto, 2009. Disponível em: <https://www.cs.toronto.edu/~kriz/learning-features-2009-TR.pdf>.
4. TENSORFLOW. **tf.keras.datasets.cifar10.load_data**. Disponível em: <https://www.tensorflow.org/api_docs/python/tf/keras/datasets/cifar10/load_data>. Acesso em: 29 jul. 2026.
5. TENSORFLOW. **tf.keras.layers.Conv2D**. Disponível em: <https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D>. Acesso em: 29 jul. 2026.
6. TENSORFLOW. **Data augmentation**. Disponível em: <https://www.tensorflow.org/tutorials/images/data_augmentation>. Acesso em: 29 jul. 2026.


## 12. Declaração de uso de IA

Ferramentas de inteligência artificial generativa da OpenAI foram utilizadas como apoio na identificação de erros de execução no Google Colab, na organização dos códigos, na revisão da redação e na estruturação do relatório. A geração de imagens também foi utilizada para criar quatro estímulos padronizados do mesmo automóvel nas condições normal, iluminação alterada, fundo complexo e mudança de escala; essas imagens foram exibidas no monitor durante os testes com webcam.

O carregamento do CIFAR-10, o treinamento dos modelos, a captura dos frames, a obtenção das métricas, a inspeção dos resultados e a validação do conteúdo foram realizados pelo grupo. As figuras numéricas foram reconstruídas exclusivamente a partir dos valores obtidos nas execuções de 29/07/2026. Nenhum resultado experimental foi inventado. Os autores revisaram o material e assumem responsabilidade integral pelo conteúdo apresentado.
